# 10 — Cross-Method Cluster Alignment

Projects all six unsupervised clustering methods into a **shared 50-dim UMAP space**
by concatenating L2-normalised MiniLM (384-dim) and RoBERTa (768-dim) train embeddings
per article (→ 1,152-dim), then fitting one UMAP on the combined matrix.

In this shared space, every cluster centroid — regardless of which method produced it —
is comparable via cosine similarity. The result is a 96×96 similarity matrix
(6 methods × 16 clusters), from which we extract 15 pairwise 16×16 alignment heatmaps
and apply Hungarian matching to find the best 1-to-1 cluster pairing across methods.

Inputs (pre-computed, no re-running):
- `embeddings_cache/minilm_train_full_summary.npy`  — (3824, 384)
- `embeddings_cache/roberta_train_full_summary.npy` — (3824, 768)
- `results/cluster_labels_{name}.npy`               — (3824,) for each of 6 methods

Outputs:
- `results/centroid_alignment_summary.csv`  — 15-row summary table
- `results/alignment_heatmap_all.png`       — 5×3 grid of 16×16 heatmaps
- `results/centroid_scatter_2d.png`         — 2D scatter of all 96 centroids

In [1]:
import sys
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import umap
from sklearn.preprocessing import normalize
from scipy.optimize import linear_sum_assignment

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

from utils import config

# ── Config ───────────────────────────────────────────────────────────────────
METHODS = [
    "minilm_kmeans", "roberta_kmeans",
    "minilm_agglomerative", "roberta_agglomerative",
    "minilm_dec", "roberta_dec",
]
N_CLUSTERS = 16
UMAP_SHARED_DIM = 50
UMAP_VIZ_DIM = 2
STRONG_ALIGN_THRESHOLD = 0.5

# ── Load embeddings ───────────────────────────────────────────────────────────
minilm_emb = np.load(REPO_ROOT / "embeddings_cache" / "minilm_train_full_summary.npy")
roberta_emb = np.load(REPO_ROOT / "embeddings_cache" / "roberta_train_full_summary.npy")

assert minilm_emb.shape == (3824, 384), f"Unexpected MiniLM shape: {minilm_emb.shape}"
assert roberta_emb.shape == (3824, 768), f"Unexpected RoBERTa shape: {roberta_emb.shape}"

# L2-normalise each before concatenating so neither family dominates by scale
minilm_norm = normalize(minilm_emb.astype(np.float32))   # (3824, 384)
roberta_norm = normalize(roberta_emb.astype(np.float32)) # (3824, 768)
combined = np.hstack([minilm_norm, roberta_norm])         # (3824, 1152)
assert combined.shape == (3824, 1152), f"Unexpected combined shape: {combined.shape}"

# ── Load cluster labels ───────────────────────────────────────────────────────
labels = {}
for name in METHODS:
    path = REPO_ROOT / "results" / f"cluster_labels_{name}.npy"
    assert path.exists(), f"Missing: {path} — run notebook 02 first"
    arr = np.load(path).astype(int)
    assert arr.shape == (3824,), f"{name}: unexpected label shape {arr.shape}"
    assert arr.min() >= 0 and arr.max() < N_CLUSTERS, \
        f"{name}: label values out of range [0, {N_CLUSTERS})"
    labels[name] = arr

print(f"Combined shape: {combined.shape}")
print(f"Loaded labels for: {list(labels.keys())}")

Combined shape: (3824, 1152)
Loaded labels for: ['minilm_kmeans', 'roberta_kmeans', 'minilm_agglomerative', 'roberta_agglomerative', 'minilm_dec', 'roberta_dec']


In [2]:
print("Fitting shared UMAP on 3824 × 1152 matrix …")
reducer = umap.UMAP(
    n_components=UMAP_SHARED_DIM,
    metric="cosine",
    random_state=42,
    verbose=True,
)
shared = reducer.fit_transform(combined)  # (3824, 50)
assert shared.shape == (3824, UMAP_SHARED_DIM), f"Unexpected shared shape: {shared.shape}"
print(f"Shared UMAP complete: {shared.shape}")

Fitting shared UMAP on 3824 × 1152 matrix …
UMAP(angular_rp_forest=True, metric='cosine', n_components=50, n_jobs=1, random_state=42, verbose=True)
Sat Sep 12 18:48:32 2026 Construct fuzzy simplicial set


C:\Users\ACER\OneDrive\Documents\final-project\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Sat Sep 12 18:49:08 2026 Finding Nearest Neighbors


Sat Sep 12 18:49:16 2026 Finished Nearest Neighbor Search


Sat Sep 12 18:49:23 2026 Construct embedding


Epochs completed:   0%|            0/500 [00:00]

	completed  0  /  500 epochs
	completed  50  /  500 epochs


	completed  100  /  500 epochs
	completed  150  /  500 epochs
	completed  200  /  500 epochs
	completed  250  /  500 epochs
	completed  300  /  500 epochs
	completed  350  /  500 epochs
	completed  400  /  500 epochs
	completed  450  /  500 epochs
Sat Sep 12 18:49:37 2026 Finished embedding


Shared UMAP complete: (3824, 50)
